In [1]:
import pandas as pd
import numpy as np

# Load the cross-validation outputs
perf_df = pd.read_csv("detailed_fold_performance.csv")
imp_df = pd.read_csv("detailed_feature_importances.csv")

# =====================================================================
# 1. COMPREHENSIVE PERFORMANCE REPORT (Across Folds) - 2 DECIMAL PLACES
# =====================================================================
print("="*120)
print("📊 PERFORMANCE SUMMARY REPORT (Aggregated across 5 Folds)")
print("="*120)

metrics = ['AUROC', 'Sensitivity', 'Specificity', 'Precision', 'F1_Score']

# Group by Model and compute Mean, SD, Min, and Max statistics
summary_perf_stats = perf_df.groupby('Model')[metrics].agg(['mean', 'std', 'min', 'max'])

# Create a clean dataframe for standard terminal readout showing only Mean ± SD
formatted_perf = pd.DataFrame(index=summary_perf_stats.index)
for m in metrics:
    formatted_perf[m] = summary_perf_stats.apply(
        lambda row: f"{row[(m, 'mean')]:.2f} ± {row[(m, 'std')]:.2f}", axis=1
    )

# Extract typical feature counts and parameters per model for context
features_summary = perf_df.groupby('Model').agg({
    'Features_After_Lasso': 'mean',
    'Optimal_Parameters': lambda x: sorted(x.value_counts().items(), key=lambda v: v[1], reverse=True)[0][0]
})

formatted_perf['Avg_Features'] = features_summary['Features_After_Lasso'].round(1)
formatted_perf['Mode_Params'] = features_summary['Optimal_Parameters']

# Set the index name exactly as shown in your layout template
formatted_perf.index.name = 'Model'

# Display wide print to prevent row-wrapping in standard consoles
pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)
print(formatted_perf.to_string())
print("\n")

# Process the background stats data frame for the CSV export
csv_perf = pd.DataFrame(index=summary_perf_stats.index)
for m in metrics:
    # Round all calculated matrix boundaries to 2 decimal places before saving
    csv_perf[f"{m}_Mean"] = summary_perf_stats[(m, 'mean')].round(2)
    # csv_perf[f"{m}_Std"] = summary_perf_stats[(m, 'std')].round(2)
    # csv_perf[f"{m}_Min"] = summary_perf_stats[(m, 'min')].round(2)
    # csv_perf[f"{m}_Max"] = summary_perf_stats[(m, 'max')].round(2)

# Append remaining design context tracking attributes
csv_perf['Avg_Features_After_Lasso'] = features_summary['Features_After_Lasso'].round(1).values
csv_perf['Mode_Optimal_Parameters'] = features_summary['Optimal_Parameters'].values
csv_perf = csv_perf.reset_index()

# Export clean performance matrix
csv_perf.to_csv("summary_model_performance.csv", index=False)
print("📝 Exported structured performance stats (rounded to 2DP) to 'summary_model_performance.csv'")


# =====================================================================
# 2. TOP 20 STABLE FEATURES PER MODEL (Aggregated Across Folds)
# =====================================================================
print("\n" + "="*120)
print("🧬 TOP 20 MUTATION PREDICTORS PER ARCHITECTURE (Ranked by Mean Importance)")
print("="*120)

top_20_per_model = {}
top_20_list = []

# Compute the mean and stability of variant weights across the 5 folds
mean_imp = imp_df.groupby(['Model', 'Feature'])['Importance_Score'].agg(['mean', 'std', 'count']).reset_index()
mean_imp.columns = ['Model', 'Feature', 'Mean_Importance', 'Std_Importance', 'Fold_Count']

for model in mean_imp['Model'].unique():
    model_data = mean_imp[mean_imp['Model'] == model]
    top_20 = model_data.sort_values(by='Mean_Importance', ascending=False).head(50).copy()
    top_20['Rank'] = range(1, len(top_20) + 1)
    
    top_20_per_model[model] = set(top_20['Feature'].tolist())
    top_20_list.append(top_20)
    
    print(f"\n💡 Model: {model}")
    print("-" * 65)
    for row in top_20.itertuples():
        print(f" {row.Rank:2d}. {row.Feature:<10} | Mean Cross-Fold Score: {row.Mean_Importance:.6f} (± {row.Std_Importance:.6f})")

# Export all Top 20 features across all models to a clean dataframe
df_top_20_features = pd.concat(top_20_list, ignore_index=True)
df_top_20_features = df_top_20_features[['Model', 'Rank', 'Feature', 'Mean_Importance', 'Std_Importance', 'Fold_Count']]
df_top_20_features.to_csv("top_20_features_per_model_nodrms.csv", index=False)
print("\n📝 Exported comprehensive model variant footprints to 'top_20_features_per_model.csv'")


# =====================================================================
# 3. INTERSECTION ANALYSIS ACROSS ALL MODELS
# =====================================================================
print("\n" + "="*120)
print("🤝 CONSENSUS BIOMARKERS: INTERSECTION OF TOP PREDICTORS")
print("="*120)

consensus_records = []

# Total intersection across absolutely all models that output weights/importance
valid_models = [m for m, feats in top_20_per_model.items() if len(feats) > 0]
all_model_features = [top_20_per_model[m] for m in valid_models]

if all_model_features:
    global_intersection = set.intersection(*all_model_features)
    print(f"\n🎯 Variants appearing in the Top 20 of ALL evaluated models ({len(global_intersection)} total):")
    print("-" * 85)
    if global_intersection:
        sorted_global = sorted(list(global_intersection))
        print(", ".join(sorted_global))
        for feat in sorted_global:
            consensus_records.append({'Feature': feat, 'Consensus_Type': 'Universal Top-20 (All Models)'})
    else:
        print("None. (No single mutation captured a top-20 ranking across every single algorithm layout).")

# Key Pairwise Intersection: Linear Models vs. Tree Ensembles
linear_group = {'Logistic Regression', 'Linear SVM'}
tree_group = {'Random Forest', 'GBM', 'XGBoost'}

available_linear = linear_group.intersection(valid_models)
available_trees = tree_group.intersection(valid_models)

if available_linear and available_trees:
    linear_union = set.intersection(*[top_20_per_model[m] for m in available_linear])
    tree_union = set.intersection(*[top_20_per_model[m] for m in available_trees])
    
    cross_method_intersection = linear_union.intersection(tree_union)
    print(f"\n🔬 High-Confidence Consensus Variants (Intersection between Linear & Tree Ensemble Top Predictors):")
    print("-" * 85)
    if cross_method_intersection:
        sorted_cross = sorted(list(cross_method_intersection))
        print(", ".join(sorted_cross))
        for feat in sorted_cross:
            if not any(r['Feature'] == feat for r in consensus_records):
                consensus_records.append({'Feature': feat, 'Consensus_Type': 'Cross-Family (Linear vs Tree Ensembles)'})
    else:
        print("None.")

# Export consensus logs
df_consensus = pd.DataFrame(consensus_records)
if df_consensus.empty:
    df_consensus = pd.DataFrame(columns=['Feature', 'Consensus_Type'])
df_consensus.to_csv("consensus_biomarkers.csv", index=False)
print("\n📝 Exported overlapping consensus biomarkers to 'consensus_biomarkers.csv'")

📊 PERFORMANCE SUMMARY REPORT (Aggregated across 5 Folds)
                           AUROC  Sensitivity  Specificity    Precision     F1_Score  Avg_Features                                  Mode_Params
Model                                                                                                                                          
Decision Tree        0.85 ± 0.01  0.70 ± 0.05  0.88 ± 0.03  0.86 ± 0.03  0.77 ± 0.03         933.8                            {'max_depth': 10}
GBM                  0.90 ± 0.00  0.74 ± 0.01  0.86 ± 0.00  0.85 ± 0.00  0.79 ± 0.00         933.8  {'learning_rate': 0.1, 'n_estimators': 100}
Linear SVM           0.90 ± 0.00  0.85 ± 0.01  0.82 ± 0.01  0.84 ± 0.01  0.84 ± 0.00         933.8                                  {'C': 0.01}
Logistic Regression  0.90 ± 0.00  0.85 ± 0.01  0.82 ± 0.01  0.84 ± 0.01  0.84 ± 0.00         933.8                                   {'C': 0.1}
Naive Bayes          0.63 ± 0.00  0.98 ± 0.00  0.27 ± 0.01  0.59 ± 0.00  0.74 ±

In [2]:
print(linear_union)

{'M50L', 'V165L', 'G134N', 'I217P', 'Q274S', 'S255Q', 'T112V', 'Q221L', 'G70E', 'Q216K', 'K236P', 'E198Q', 'D232K', 'I141V', 'D278A', 'R284M', 'D41X', 'G106X', 'P90A', 'I204V', 'G149R', 'I251F', 'T218S', 'E35Q', 'I220N', 'P233L', 'E287L', 'R224X', 'Q252K', 'S283G', 'M178S', 'D3L', 'E11D', 'I60A', 'G82K', 'H171S', 'Y83F'}


In [3]:
# Model: Logistic Regression
TN=16221
FP=3541
FN=3226
TP=18259
lr_acc = (TP+TN)/(TP+TN+FP+FN)
print(round(lr_acc,2))

0.84


In [4]:
# Model: Linear SVM
TN=16184
FP=3578
FN=3306
TP=18179
svm_acc = (TP+TN)/(TP+TN+FP+FN)
print(round(svm_acc,2))

0.83


In [5]:
# Model: Decision Tree
TN=17366
FP=2396
FN=6371
TP=15114
dt_acc = (TP+TN)/(TP+TN+FP+FN)
print(round(dt_acc,2))

0.79


In [6]:
# Model: Random Forest
TN=17171
FP=2591
FN=5924
TP=15561
rf_acc = (TP+TN)/(TP+TN+FP+FN)
print(round(rf_acc,2))

0.79


In [7]:
# Model: GBM
TN=16967
FP=2795
FN=5528
TP=15957
gbm_acc = (TP+TN)/(TP+TN+FP+FN)
print(round(gbm_acc,2))

0.8


In [8]:
# Model: XGBoost
TN=17209
FP=2553
FN=4619
TP=16866
xgb_acc = (TP+TN)/(TP+TN+FP+FN)
print(round(xgb_acc,2))

0.83


In [9]:
# Model: Naive Bayes
TN=5286
FP=14476
FN=334
TP=21151
nb_acc = (TP+TN)/(TP+TN+FP+FN)
print(round(nb_acc,2))

0.64
